# geo0646 – DWD Climate Stations: from a Text File on the Web to a CSV for QGIS

**Unit:** geo0640 DWD CDC Stations & Time Series

The German Weather Service (DWD) publishes the list of its climate stations as a plain text file.
In this notebook you turn that file into a clean CSV table which you then open in QGIS as a point layer.

```
DWD Open Data (HTTPS)  →  raw text  →  column names  →  pandas DataFrame  →  quality checks  →  CSV  →  QGIS
       download            look at it    from line 1       read_fwf            check, fix        save     points on a map
```

**Learning goals.** After this notebook you can
1. download an open dataset via HTTPS and keep a local copy,
2. look at an unknown text file and describe its format (columns, encoding, dates),
3. build column names from a header line with `split()` and `strip()` and translate them with a dictionary,
4. read a fixed-width file with `pd.read_fwf` and **check** the result instead of trusting it,
5. select rows with boolean masks,
6. write a CSV that QGIS can turn into points (coordinates in **EPSG:4326**, WGS 84 longitude/latitude).

**You need:** the pandas you know from Scientific Programming. Nothing new to install.

**Deliverable:** two CSV files in `data/derived/` and a QGIS layer of the stations (section 9).

**Exercises** are marked with ✏️. Replace `# YOUR CODE HERE` with your code and run the cell.

## 0. Setup

In [ ]:
from pathlib import Path

import requests
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print("pandas", pd.__version__)

In [ ]:
# Folders: original downloads are never edited, everything we produce goes to data/derived
ORIG_DIR = Path("data/original/dwd/annual/kl/historical")
OUT_DIR = Path("data/derived")
ORIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = ("https://opendata.dwd.de/climate_environment/CDC/"
            "observations_germany/climate/annual/kl/historical/")
STATION_FILE = "KL_Jahreswerte_Beschreibung_Stationen.txt"

## 1. Download: once, then work with the local copy
Open the [DWD directory](https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/annual/kl/historical/)
in your browser. It is a plain list of files. Scroll to `KL_Jahreswerte_Beschreibung_Stationen.txt` and click it:
this is the station list of the annual climate data (KL = *Klima*).

The function below downloads a file only if we do not have it yet. Run it twice: the second time it says `cached`.

In [ ]:
def fetch(url, dest):
    # Download url to the file dest, unless dest already exists. Returns dest.
    if dest.exists():
        print(f"cached: {dest}")
        return dest
    r = requests.get(url, timeout=60)
    r.raise_for_status()                 # stop with an error if the server says e.g. 404 Not Found
    dest.write_bytes(r.content)
    print(f"downloaded {len(r.content):,} bytes -> {dest}")
    return dest


station_path = fetch(BASE_URL + STATION_FILE, ORIG_DIR / STATION_FILE)
station_path

## 2. Look at the file before you read it
**Predict:** Is there a separator like `,` or `;`? How are dates written? Which special characters do you expect in German station names?

We read the first lines as text. The file is not UTF-8 but **Latin-1** (ISO-8859-1), an older encoding for Western European characters.

In [ ]:
with open(station_path, encoding="latin-1") as f:
    lines = f.readlines()

print(len(lines), "lines")
for line in lines[:5]:
    print(repr(line[:150]))

What we see:
* Line 1: the **column names**, separated by single blanks.
* Line 2: a row of dashes (decoration, no data).
* From line 3: the data in **fixed-width columns**: every value starts at the same position in each line; the gaps are filled with blanks.
  There is no separator, and station names may contain blanks themselves (`Bergzabern, Bad`).
* Dates are written as `YYYYMMDD`, e.g. `19310101`.
* `\r\n` at the end: Windows line endings.

✏️ **Try it:** read the file once more with `encoding="utf-8"` instead of `latin-1`. What happens, and why?

In [ ]:
# YOUR TRY HERE (optional)

## 3. Column names from the first line
The header line is one string. We want a **list of clean names** that we can pass to pandas.

Reminder (string methods):
* `"a  b c".split()` → `['a', 'b', 'c']`: splits at any run of whitespace,
* `" Abc\r\n".strip()` → `'Abc'`: removes whitespace (also `\r`, `\n`) at both ends,
* `"Abc".lower()` → `'abc'`.

✏️ **Exercise 3.1:** create the list `names` from `lines[0]`: split the header, then **strip** and **lower-case** every name
(a list comprehension is the shortest way).

In [ ]:
header = lines[0]
print(repr(header))

# YOUR CODE HERE
names

In [ ]:
# Check: run this cell. It must print "ok".
assert names[0] == "stations_id" and names[1] == "von_datum" and "stationsname" in names, names
print("ok:", len(names), "columns")

## 4. Read the data with `pd.read_fwf`
`read_fwf` = read **f**ixed-**w**idth **f**ormatted lines. If we do not tell pandas where the columns start and end, it **guesses** the column borders from the data.

✏️ **Exercise 4.1:** read the file into the DataFrame `df` with
* `skiprows=2` (header and dashes; we give the names ourselves),
* `names=names`,
* `encoding="latin-1"`,
* `index_col="stations_id"` (the station id becomes the row label),
* `parse_dates=["von_datum", "bis_datum"]` (turn `19310101` into a date).

Pattern: `pd.read_fwf(<file>, skiprows=2, encoding="latin-1", names=..., index_col=..., parse_dates=[...])`

In [ ]:
# YOUR CODE HERE
df.head()

In [ ]:
df.dtypes

Looks fine: dates are dates, altitude is an integer, coordinates are floats. **But is it right?**

## 5. Quality checks: never skip this step
### 5.1 Overview
How many stations? Missing values? Plausible ranges? Germany lies roughly between 47 and 55.1 °N and 5.8 and 15.1 °E.

In [ ]:
print(len(df), "stations")
print(df.isna().sum())
df[["stationshoehe", "geobreite", "geolaenge"]].describe()

The empty values in `abgabe` (data access, *Frei* = free) are not a problem for us: the column is simply empty for a few stations.

### 5.2 The highest stations
**Predict:** the highest DWD station is on the **Zugspitze**, Germany's highest mountain (2 962 m). What will the table say?

✏️ **Exercise 5.2:** show the 10 highest stations (`sort_values` on `stationshoehe`, descending, `head(10)`).
Then find the Zugspitze with `df[df["stationsname"] == "Zugspitze"]`.

In [ ]:
# YOUR CODE HERE
top10[["stationsname", "stationshoehe", "bundesland"]]

In [ ]:
df[df["stationsname"] == "Zugspitze"]

**956 m for the Zugspitze?** Something is wrong, although pandas did not complain at all.

Let us compare the raw line of the Zugspitze with the line of the first station:

In [ ]:
for line in lines:
    if line.startswith("00001") or "Zugspitze " in line:
        print(repr(line[:120]))

In the raw file the value is `2956`. The digit `2` is the problem: `read_fwf` guesses the column borders from the **first 100 data lines only**
(parameter `infer_nrows=100`). In those lines no altitude has 4 digits and no name is very long, so pandas places the borders too tight.
Later lines with 4-digit altitudes or long names are **cut silently**.

**Fix:** let pandas look at *all* lines when it guesses the borders: `infer_nrows=len(lines)`.

✏️ **Exercise 5.3:** read the file again into `df_ok`, same parameters as in 4.1 plus `infer_nrows=len(lines)`.

In [ ]:
# YOUR CODE HERE
df_ok[df_ok["stationsname"] == "Zugspitze"]

✏️ **Exercise 5.4:** how many stations had a wrong altitude in `df`? Compare the two columns with `!=` and count the `True` values with `.sum()`.
Then show the wrong and the correct value side by side.

In [ ]:
# YOUR CODE HERE
pd.DataFrame({"name": df_ok["stationsname"], "wrong": df["stationshoehe"], "correct": df_ok["stationshoehe"]})[wrong]

In [ ]:
# From now on we work with the corrected table
df = df_ok
df.sort_values("stationshoehe", ascending=False).head(5)[["stationsname", "stationshoehe"]]

> **Lesson:** a reader that does not crash is not necessarily a reader that is right. Check the result against something you know.

### 5.5 Dates
A station should not stop measuring before it started.

✏️ **Exercise 5.5:** select all stations with `von_datum` later than `bis_datum` into `bad_dates`. How many are there?

In [ ]:
# YOUR CODE HERE
print(len(bad_dates), "stations with von_datum > bis_datum")
bad_dates[["stationsname", "von_datum", "bis_datum", "bundesland"]]

**Discuss:** these are errors in the source file (or stations announced but not yet delivering data). We **keep** them in the table,
but we do not count them as long-running stations. In a report you would tell the data provider.

## 6. English column names with a dictionary
The column names come from the DWD file and are German. For an international team (and for QGIS field names) we translate them.
We do **not** edit the original file: the translation happens in code, so it is documented and repeatable.

A **dictionary** maps each German name (*key*) to an English name (*value*):
`{"stations_id": "station_id", ...}`. Then `df.rename(columns=<dictionary>)` renames all columns in one go.

Vocabulary:

| German (in the file) | meaning | English column name |
|---|---|---|
| `stations_id` | station identifier | `station_id` |
| `von_datum` | date from (first measurement) | `date_from` |
| `bis_datum` | date to (last measurement) | `date_to` |
| `stationshoehe` | station altitude (*Höhe*) in m | `altitude` |
| `geobreite` | geographic latitude (*Breite*), y | `lat` |
| `geolaenge` | geographic longitude (*Länge*), x | `lon` |
| `stationsname` | station name | `name` |
| `bundesland` | federal state | `state` |
| `abgabe` | data access (*Frei* = free) | `access` |

✏️ **Exercise 6.1:** create the dictionary `COLUMNS_DE_EN` with one entry for each of the 9 names in `names`. Use the English names of the table.

In [ ]:
print(names)

# COLUMNS_DE_EN = {
#     "stations_id": "station_id",
#     "von_datum": ...,
#     ...
# }
# YOUR CODE HERE
COLUMNS_DE_EN

In [ ]:
# Check: run this cell. It must print "ok".
missing = [n for n in names if n not in COLUMNS_DE_EN]
assert not missing, f"not translated yet: {missing}"
assert COLUMNS_DE_EN["geobreite"] == "lat" and COLUMNS_DE_EN["geolaenge"] == "lon", "lat/lon swapped? Breite = latitude (y), Laenge = longitude (x)"
assert len(set(COLUMNS_DE_EN.values())) == len(COLUMNS_DE_EN), "two German names get the same English name"
print("ok:", len(COLUMNS_DE_EN), "translations")

✏️ **Exercise 6.2:** rename the columns of `df` with the dictionary and store the result in `df` again.
The station id is the **index**, not a column, so `rename(columns=...)` does not touch it. Give the index its English name with
`.rename_axis(COLUMNS_DE_EN["stations_id"])`.

In [ ]:
# YOUR CODE HERE
assert df.index.name == "station_id" and "altitude" in df.columns, df.columns
df.head()

**Discuss:**
* Which mistake does the `assert` on `lat`/`lon` protect you from, and what would it look like on the map in QGIS?
* What happens if a key of the dictionary is **not** a column of the DataFrame? Try `df.rename(columns={"xyz": "abc"})`.

## 7. Select stations with boolean masks
A comparison on a column gives a Series of `True`/`False` (a *mask*). Combine masks with `&` (and), `|` (or), and put each condition in brackets.

✏️ **Exercise 7.1:** select all stations in North Rhine-Westphalia (`state` is `"Nordrhein-Westfalen"`) into `df_nrw`. How many are there?

In [ ]:
# YOUR CODE HERE
print(len(df_nrw), "stations in NRW")

✏️ **Exercise 7.2:** a station is **active** if its `date_to` equals the most recent end date in the whole file (`df["date_to"].max()`).
Select the active NRW stations into `df_nrw_active`.

In [ ]:
latest = df["date_to"].max()
print("latest end date:", latest.date())

# YOUR CODE HERE
print(len(df_nrw_active), "active stations in NRW")
df_nrw_active[["name", "altitude", "date_from"]]

✏️ **Exercise 7.3:** answer with code:
1. Which active NRW station is the highest, which the lowest?
2. How many active NRW stations started measuring in **1950 or earlier**? (Hint: `df_nrw_active["date_from"].dt.year`.)

In [ ]:
# YOUR CODE HERE


## 8. Save as CSV for QGIS
QGIS reads CSV files as point layers if it knows **which columns hold the coordinates** and **in which CRS** they are.
Our coordinates are geographic longitude (`lon`, x) and latitude (`lat`, y) in degrees, WGS 84: **EPSG:4326**.
We do not convert them; we just write them out unchanged.

Choices for the file:
* **separator `;`**: station names contain commas (`Bergzabern, Bad`), so a comma separator would need quotes,
* **encoding UTF-8**: the standard today (QGIS detects it),
* **dates as `YYYY-MM-DD`** (ISO 8601): sortable as text, recognised by QGIS as dates,
* the index `station_id` is written as the first column.

✏️ **Exercise 8.1:** write `df` to `OUT_DIR / "dwd_kl_stations.csv"` and `df_nrw_active` to `OUT_DIR / "dwd_kl_stations_nrw_active.csv"`
using `to_csv(<file>, sep=";", encoding="utf-8", date_format="%Y-%m-%d")`.

In [ ]:
csv_all = OUT_DIR / "dwd_kl_stations.csv"
csv_nrw = OUT_DIR / "dwd_kl_stations_nrw_active.csv"

# YOUR CODE HERE
print(csv_all, csv_all.stat().st_size, "bytes")
print(csv_nrw, csv_nrw.stat().st_size, "bytes")

**Check** the result as text (this is what QGIS will see) and by reading it back:

In [ ]:
with open(csv_all, encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().rstrip())

check = pd.read_csv(csv_all, sep=";", index_col="station_id", parse_dates=["date_from", "date_to"])
assert len(check) == len(df) and check.loc[5792, "altitude"] == 2956
print("ok:", len(check), "stations, Zugspitze", check.loc[5792, "altitude"], "m")

## 9. QGIS: the CSV as a point layer
1. In JupyterLab, right-click `data/derived/dwd_kl_stations.csv` → **Download**. Do the same for `dwd_kl_stations_nrw_active.csv`.
2. In QGIS set the project CRS to **EPSG:25832** (bottom right), add an OpenStreetMap basemap (*XYZ Tiles*).
3. *Layer → Add Layer → Add Delimited Text Layer…*
   * **File format:** Custom delimiters, tick **Semicolon**,
   * **Record and Fields Options:** *First record has field names*, *Detect field types*,
   * **Geometry definition:** *Point coordinates*, **X field `lon`**, **Y field `lat`**,
   * **Geometry CRS: EPSG:4326 – WGS 84**,
   * *Add*. QGIS reprojects the points on the fly into the project CRS.
4. **Check:** does *Aachen* (id 3) sit on Aachen? Open the attribute table: are `date_from`/`date_to` recognised as dates, `altitude` as integer?
5. **Check:** symbolise the layer *Graduated* on `altitude`. Are the high stations in the Alps, the Black Forest, the Harz?
6. **Find the odd one:** one station lies in the open sea. Which one, and which federal state does the file give for it?
7. Load `dwd_kl_stations_nrw_active.csv` the same way and save it as a GeoPackage in EPSG:25832
   (*right-click → Export → Save Features As…*, format GeoPackage, CRS EPSG:25832). Why is a GeoPackage better than a CSV for further work?

**Check yourself:** what happens if you choose EPSG:25832 as *Geometry CRS* for this CSV in step 3? Try it, zoom to the layer and explain.

## 10. Exercise in pairs: another DWD station list
The hourly precipitation network has its own station list:

`https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/hourly/precipitation/recent/RR_Stundenwerte_Beschreibung_Stationen.txt`

✏️ **Exercise 10.1:**
1. Download it with `fetch()` into `data/original/dwd/hourly/precipitation/recent/`.
2. Look at the first lines. Same columns as the KL list? (Do not assume, check!)
3. Build the names from the header (as in §3) and read it with `read_fwf` (as in §5.3, with `infer_nrows`).
4. Translate the column names with your dictionary `COLUMNS_DE_EN` (§6). Does it still work although a column is missing?
5. Do the quality checks of §5 (number of stations, missing values, highest station).
6. Select the **active** precipitation stations in NRW and write them to `OUT_DIR / "dwd_rr_stations_nrw_active.csv"`.
7. In QGIS: load it as a second point layer. Which network is denser in NRW, climate (KL) or precipitation (RR)?

In [ ]:
# YOUR CODE HERE


**Discuss:** why was it a good idea to read the column names from the header line instead of typing them?

## 11. Stretch (optional): one function for all DWD station lists
✏️ Write a function `read_dwd_stations(path)` that does §2–§6 in one go (read lines, build names, `read_fwf` with `infer_nrows`, English names) and returns the DataFrame.
Test it with two `assert` statements, e.g. that the station ids are unique and that all latitudes are between 47 and 56.

In [ ]:
# YOUR CODE HERE


## 12. Wrap-up
* **Inspect** a file before you read it: separator, encoding, header lines, date format.
* **Take names from the data** (header line) instead of typing them; translate them with a **dictionary** in code, never by editing the original file.
* **Check the result** against something you know (the Zugspitze): a silent error is worse than a crash.
* A CSV becomes a map layer in QGIS when you name the **x/y columns** and their **CRS** (here EPSG:4326).

**Exit ticket (2 min):** what was the *muddiest point* today? One sentence in the course chat or on a card.

**Homework and presentations:** see the course plan in the README of your course repository.